## 1. Dataset

In [188]:
# pull down the datasets from kaggle:
import kagglehub

path_1 = kagglehub.dataset_download("uom190346a/global-coffee-health-dataset")
print("Path to dataset files:", path_1)

path_2 = kagglehub.dataset_download("heitornunes/caffeine-content-of-drinks")
print("Path to dataset files:", path_2)

Path to dataset files: /Users/shujiazhang/.cache/kagglehub/datasets/uom190346a/global-coffee-health-dataset/versions/1
Path to dataset files: /Users/shujiazhang/.cache/kagglehub/datasets/heitornunes/caffeine-content-of-drinks/versions/1


## 2. Store dataset into database:
* **Coffee health** dataset into SQL
* **caffeine** dataset into MongoDB

### 2.1. SQL database:

In [ ]:
# connect to SQL localhost through docker:
import os
import psycopg

conn = psycopg.connect(
    host="localhost",
    port="5432",
    dbname="my_db",
    user="postgres",
    password=os.environ["DB_PASSWORD"]
)

cur = conn.cursor()

Connecting to the PostgreSQL database...


In [190]:
# load Coffee Health into pandas dataframe:
import pandas as pd

csv_path_1 = [f for f in os.listdir(path_1) if f.endswith(".csv")][0]
full_csv_path_1 = os.path.join(path_1, csv_path_1)

health_full = pd.read_csv(full_csv_path_1)

In [191]:
# create database in SQL:
cur.execute("DROP TABLE IF EXISTS coffee_health_full;")

create_table = """
               CREATE TABLE coffee_health_full (
               id INT,
               age FLOAT,
               gender TEXT,
               country TEXT,
               coffee_intake FLOAT,
               caffeine_mg FLOAT,
               sleep_hours FLOAT,
               sleep_quality TEXT,
               bmi FLOAT,
               heart_rate FLOAT,
               stress_level TEXT,
               physical_activity_hours FLOAT,
               health_issues TEXT,
               occupation TEXT,
               smoking INT,
               alcohol_consumption INT
               );
               """

cur.execute(create_table)
conn.commit()
print("Table created successfully.")

Table created successfully.


In [192]:
# store data in SQL:
insert_query = """
               INSERT INTO coffee_health_full (
               id, age, gender, country, coffee_intake, caffeine_mg, sleep_hours, sleep_quality, bmi, heart_rate, 
               stress_level, physical_activity_hours, health_issues, occupation, smoking, alcohol_consumption
               )
               VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
               """

for _, row in health_full.iterrows():
    cur.execute(insert_query, (
        int(row["ID"]) if pd.notna(row["ID"]) else None,
        float(row["Age"]) if pd.notna(row["Age"]) else None,
        str(row["Gender"]) if pd.notna(row["Gender"]) else None,
        str(row["Country"]) if pd.notna(row["Country"]) else None,
        float(row["Coffee_Intake"]) if pd.notna(row["Coffee_Intake"]) else None,
        float(row["Caffeine_mg"]) if pd.notna(row["Caffeine_mg"]) else None,
        float(row["Sleep_Hours"]) if pd.notna(row["Sleep_Hours"]) else None,
        str(row["Sleep_Quality"]) if pd.notna(row["Sleep_Quality"]) else None,
        float(row["BMI"]) if pd.notna(row["BMI"]) else None,
        float(row["Heart_Rate"]) if pd.notna(row["Heart_Rate"]) else None,
        str(row["Stress_Level"]) if pd.notna(row["Stress_Level"]) else None,
        float(row["Physical_Activity_Hours"]) if pd.notna(row["Physical_Activity_Hours"]) else None,
        str(row["Health_Issues"]) if pd.notna(row["Health_Issues"]) else None,
        str(row["Occupation"]) if pd.notna(row["Occupation"]) else None,
        int(row["Smoking"]) if pd.notna(row["Smoking"]) else None,
        int(row["Alcohol_Consumption"]) if pd.notna(row["Alcohol_Consumption"]) else None
    ))

conn.commit()
print("Data inserted successfully into coffee_health_full.")

Data inserted successfully into coffee_health_full.


In [193]:
cur.execute("DROP TABLE IF EXISTS coffee_health;")

clean_sql = """
            CREATE TABLE coffee_health AS
            SELECT
            Age,
            BMI,
            Stress_Level,
            Gender,
            Caffeine_mg,
            Heart_Rate,
            Sleep_Hours
            FROM coffee_health_full
            WHERE
            Age IS NOT NULL AND
            BMI IS NOT NULL AND
            Stress_Level IS NOT NULL AND
            Gender IS NOT NULL AND
            Caffeine_mg IS NOT NULL AND
            Heart_Rate IS NOT NULL AND
            Sleep_Hours IS NOT NULL;
            """

cur.execute(clean_sql)
conn.commit()
print("Cleaned table 'coffee_health' created successfully")

Cleaned table 'coffee_health' created successfully


In [194]:
health_df = pd.read_sql("SELECT * FROM coffee_health;", conn)
print("Data cleaning and selection done")
health_df.head()

Data cleaning and selection done


/var/folders/t7/fhrwqn1s6810tvdnfvsd69g00000gn/T/ipykernel_35483/283039991.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  health_df = pd.read_sql("SELECT * FROM coffee_health;", conn)


,age,bmi,stress_level,gender,caffeine_mg,heart_rate,sleep_hours
0,40.0,24.9,Low,Male,328.1,78.0,7.5
1,33.0,20.0,Low,Male,94.1,67.0,6.2
2,42.0,22.7,Medium,Male,503.7,59.0,5.9
3,53.0,24.7,Low,Male,249.2,71.0,7.3
4,32.0,24.1,Medium,Female,298.0,76.0,5.3


### 2.2. MongoDB database:

In [195]:
# connect to mongoDB:
from pymongo import MongoClient

client = MongoClient('localhost', 27017)
db = client['apan5400']
collection = db["caffeine_content"]

In [196]:
csv_file_2 = [f for f in os.listdir(path_2) if f.endswith(".csv")][0]
df_drinks = pd.read_csv(os.path.join(path_2, csv_file_2))

In [197]:
# convert and insert dataset into MongoDB + data cleaning:
df_drinks.rename(columns = lambda x: x.strip(), inplace = True)
data_dict = df_drinks.to_dict("records")

collection.delete_many({})
collection.insert_many(data_dict)
print(f"Successfully inserted {len(data_dict)} records into MongoDB collection 'caffeine_content'")

Successfully inserted 610 records into MongoDB collection 'caffeine_content'


In [198]:
# verify storage in MongoDB:
for doc in collection.find().limit(5):
    print(doc)

{'_id': ObjectId('69227af97c050f30de0d68fb'), 'drink': 'Costa Coffee', 'Volume (ml)': 256.99371499999995, 'Calories': 0, 'Caffeine (mg)': 277, 'type': 'Coffee'}
{'_id': ObjectId('69227af97c050f30de0d68fc'), 'drink': 'Coffee Friend Brewed Coffee', 'Volume (ml)': 250.19181000000003, 'Calories': 0, 'Caffeine (mg)': 145, 'type': 'Coffee'}
{'_id': ObjectId('69227af97c050f30de0d68fd'), 'drink': 'Hell Energy Coffee', 'Volume (ml)': 250.19181000000003, 'Calories': 150, 'Caffeine (mg)': 100, 'type': 'Coffee'}
{'_id': ObjectId('69227af97c050f30de0d68fe'), 'drink': 'Killer Coffee (AU)', 'Volume (ml)': 250.19181000000003, 'Calories': 0, 'Caffeine (mg)': 430, 'type': 'Coffee'}
{'_id': ObjectId('69227af97c050f30de0d68ff'), 'drink': 'Nescafe Gold', 'Volume (ml)': 250.19181000000003, 'Calories': 0, 'Caffeine (mg)': 66, 'type': 'Coffee'}


In [199]:
# calculating the suggest limit of intake caffeine:
pipeline = [
    {"$group": {
        "_id": "$Category",
        "avg_caffeine": {"$avg": "$Caffeine_mg"},
        "max_caffeine": {"$max": "$Caffeine_mg"},
        "count": {"$sum": 1}
    }},
    {"$sort": {"avg_caffeine": -1}}
]
stats = list(collection.aggregate(pipeline))
pd.DataFrame(stats)

,_id,avg_caffeine,max_caffeine,count
0,None,None,None,610


In [200]:
pipeline = [
    {"$project": {
        "_id": 0,
        "Drink": 1,
        "Category": 1,
        "Caffeine_mg": 1,
        "Above_Adult_Limit": {"$cond": [{"$gt": ["$Caffeine_mg", 400]}, True, False]},
        "Above_Teen_Limit": {"$cond": [{"$gt": ["$Caffeine_mg", 100]}, True, False]},
        "Above_Senior_Limit": {"$cond": [{"$gt": ["$Caffeine_mg", 300]}, True, False]}
    }},
    {"$match": {"Above_Adult_Limit": True}}
]
results = list(collection.aggregate(pipeline))
pd.DataFrame(results).head()

""


In [201]:
db.command({
    "create": "risky_drinks",
    "viewOn": "caffeine_content",
    "pipeline": [
        {"$match": {"Caffeine_mg": {"$gt": 400}}}
    ]})

{'ok': 1.0}

### 3. Model ->  **Random Forest Regression**:

In [202]:
from flask import Flask, request, jsonify, render_template
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

In [203]:
# data preperation:
df = health_df.copy()
df = pd.get_dummies(df, columns = ["gender", "stress_level"], drop_first = True)

feature_cols = ["caffeine_mg", "age", "bmi"] + [
    c for c in df.columns if c.startswith("gender_") or c.startswith("stress_level_")
]

In [204]:
# split train and test:
X = df[feature_cols]
y_hr = df["heart_rate"]
y_sleep = df["sleep_hours"]

X_train_hr, _, y_train_hr, _ = train_test_split(
    X, y_hr, test_size = 0.2, random_state = 42
)
X_train_sl, _, y_train_sl, _ = train_test_split(
    X, y_sleep, test_size = 0.2, random_state = 42
)

In [205]:
# model trainning:
model_hr = RandomForestRegressor(
    n_estimators = 200, random_state = 42, n_jobs = -1
)
model_hr.fit(X_train_hr, y_train_hr)

model_sleep = RandomForestRegressor(
    n_estimators = 200, random_state = 42, n_jobs = -1
)
model_sleep.fit(X_train_sl, y_train_sl)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [206]:
# save the models:
import pickle
import os

os.makedirs("models", exist_ok = True)

pickle.dump(model_hr, open("models/model_hr.pkl", "wb"))
pickle.dump(model_sleep, open("models/model_sleep.pkl", "wb"))
pickle.dump(feature_cols, open("models/feature_cols.pkl", "wb"))

In [207]:
# end sql session:
cur.close()
conn.close()

In [208]:
# end MongoDB session:
client.close()